In [2]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go



/Users/parthapratimdas/RAG-basics/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/parthapratimdas/RAG-basics/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv(override=True)
MODEL = "deepseek/deepseek-r1:free"
db_name = "vector_db"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openrouter_api_key:
    print("OPENROUTER_API_KEY is set")
else:
    print("OPENROUTER_API_KEY is not set")


OPENROUTER_API_KEY is set


In [4]:
knowledge_base_path = "knowledge-base/**/*.md"

files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

Found 23 files in the knowledge base


In [5]:
entire_knowledge_base = ""

In [6]:
for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total number of characters in the knowledge base: {len(entire_knowledge_base)}")

Total number of characters in the knowledge base: 26955


In [7]:
import tiktoken

# 1. Try to get the model-specific encoding, otherwise fall back to a standard one
try:
    encoding = tiktoken.encoding_for_model(MODEL)
except KeyError:
    # Most modern models use the cl100k_base (GPT-4) or o200k_base (GPT-4o) patterns
    print(f"Warning: Model '{MODEL}' not recognized. Falling back to 'cl100k_base'.")
    encoding = tiktoken.get_encoding("cl100k_base")

# 2. Encode the knowledge base
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)

# 3. Use 'print' (printf is for C/C++)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for deepseek/deepseek-r1:free: 6,394


In [8]:
folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 23 documents


In [9]:
documents[1]

Document(metadata={'source': 'knowledge-base/products/product2-secureops-fusion.md', 'doc_type': 'products'}, page_content='# SecureOps Fusion\n\n- Product ID: PROD-ACN-002\n- Company: Accenture\n- Category: Managed Cybersecurity Platform\n- Version: 4.1\n- Launch Year: 2022\n- Deployment: Managed Service, Hybrid Cloud\n- Primary Users: CISO Office, SOC Teams, IT Risk\n- Target Industries: Energy, Healthcare, Financial Services\n\n## Overview\nSecureOps Fusion unifies security monitoring, threat detection, and incident response orchestration into a single managed platform.\n\n## Core Features\n- 24x7 SOC telemetry aggregation and triage\n- Automated response playbooks for high-confidence alerts\n- Threat intelligence correlation and attack path analysis\n- Compliance reporting for common audit frameworks\n\n## Integrations\n- Splunk\n- Microsoft Sentinel\n- Palo Alto Cortex XSOAR\n- CrowdStrike Falcon\n\n## Pricing (Synthetic)\n- Base Managed SOC: USD 420,000 per year\n- Advanced Threa

In [10]:

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 41 chunks
First chunk:

page_content='# CareAssist Voice AI

- Product ID: PROD-ACN-003
- Company: Accenture
- Category: Healthcare Contact Center AI
- Version: 2.7
- Launch Year: 2024
- Deployment: Hybrid Cloud
- Primary Users: Patient Support Teams, Contact Center Managers
- Target Industries: Healthcare Providers, Payers

## Overview
CareAssist Voice AI is a conversational AI solution for patient engagement, appointment workflows, and claims routing in healthcare support centers.

## Core Features
- Natural language voice bots for inbound patient requests
- Intelligent triage and escalation to human agents
- Appointment booking and reminder workflows
- Sentiment analysis and quality scoring

## Integrations
- Epic
- Salesforce Health Cloud
- Genesys Cloud CX
- Twilio

## Pricing (Synthetic)
- Platform License: USD 210,000 per year
- Usage: USD 0.022 per voice minute
- Clinical Workflow Pack: USD 65,000 per year' metadata={'source': 'knowledge-base/products/product3-care

In [11]:
load_dotenv(override=True)
model = "all-MiniLM-L6-v2"
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    print("hf_token is set")
else:
    print("hf_token is not set")


hf_token is set


In [12]:
 # Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 41 documents


In [13]:

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 41 vectors with 384 dimensions in the vector store


In [14]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [ ]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

/Users/parthapratimdas/RAG-basics/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/parthapratimdas/RAG-basics/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/parthapratimdas/RAG-basics/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/parthapratimdas/RAG-basics/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/parthapratimdas/RAG-basics/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/parthapratimdas/RAG-basics/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: inval

In [17]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

/Users/parthapratimdas/RAG-basics/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/parthapratimdas/RAG-basics/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/parthapratimdas/RAG-basics/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/parthapratimdas/RAG-basics/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/parthapratimdas/RAG-basics/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/parthapratimdas/RAG-basics/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: inval